# 🏨 AGODA Price Crawler

Chạy lần lượt các cell từ trên xuống: **① Cấu hình → ② Đọc input → ③ Crawl → ④ Xem kết quả**.

**Đổi nguồn input:** sửa `INPUT_MODE` ở cell ① — `"gsheet"` (Google Sheet online) hoặc `"offline"` (file CSV/XLSX trên máy).

**Format file offline:** chỉ cần **3 cột đầu theo đúng thứ tự** `hotel_name, hotel_url, room_type` (tên cột không quan trọng, chỉ cần đúng thứ tự). Có file mẫu ở `input/TEMPLATE_hotels.csv`.

**Input offline** nằm riêng theo notebook: `input/agoda-thy-6.csv` (cell ① `OFFLINE_FILE` / cell tải Sheet).

**Output** nằm trong `results/agoda/<RUN_NAME>/` — notebook này dùng `thy-6`, không ghi đè notebook khác:
- `FINAL_<YYYYMMDD>.csv` — kết quả cuối
- `TEMP_agoda.csv` — checkpoint: lỡ tắt giữa chừng, chạy lại cell ③ sẽ tự resume phần chưa xong — hotel **chưa từng cào** sẽ chạy trước, hotel còn NA/SOLD OUT retry sau

Cache warm (`results/agoda/captures/`) vẫn dùng chung giữa các notebook nên không tốn thêm thời gian warm.

In [1]:
# ════════════════ ① CẤU HÌNH ════════════════

# ── Tên run: kết quả nằm RIÊNG trong results/agoda/<RUN_NAME>/ ──
RUN_NAME = "thy-6"

# ── Nguồn input: "gsheet" (online) hoặc "offline" (file trên máy) ──
INPUT_MODE = "gsheet"

# Dùng khi INPUT_MODE = "gsheet" (gid của tab được tự lấy từ URL)
GSHEET_URL = "https://docs.google.com/spreadsheets/d/1MWYqfsbJkZoxt0iWANyI7WZN6Yq-OmbUK2SlNqK2lTc/edit?gid=1083140588#gid=1083140588"

# Dùng khi INPUT_MODE = "offline" — đường dẫn tuyệt đối, hoặc tương đối so với 31.crawl-tool
# ⚠️ File phải có 3 cột đầu là (tên KS, URL, loại phòng) — file "TEMP_*" là checkpoint OUTPUT, không phải input!
OFFLINE_FILE = "input/agoda-thy-6.csv"

# ── Tham số crawl ──
WEEKS      = 6      # số tuần cần crawl
MAX_HOTELS = 0      # 0 = crawl tất cả; đặt 5 để test nhanh 5 khách sạn đầu
SHARD      = ""     # "" = không chia; "1/3" = chạy phần 1 trong 3 phần (chạy lần lượt 1/3, 2/3, 3/3)

In [2]:
# ════════════════ ② ĐỌC INPUT ════════════════
import os, sys

if "ROOT" not in globals():                    # giữ nguyên ROOT khi chạy lại cell
    ROOT = os.path.abspath("")                 # .../31.crawl-tool (nơi đặt notebook này)
assert os.path.isdir(os.path.join(ROOT, "crawler")), (
    f"Không tìm thấy package `crawler` trong {ROOT} — hãy mở notebook từ thư mục 31.crawl-tool")
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import crawler
from crawler.hotels_io import read_hotels

if INPUT_MODE == "gsheet":
    INPUT = GSHEET_URL
    print("📡 Input: Google Sheet online")
else:
    INPUT = OFFLINE_FILE if os.path.isabs(OFFLINE_FILE) else os.path.join(ROOT, OFFLINE_FILE)
    assert os.path.exists(INPUT), f"Không tìm thấy file: {INPUT}"
    print(f"📁 Input: file offline — {INPUT}")

hotels = read_hotels(INPUT)
print(f"✅ Đọc được {len(hotels)} khách sạn. 5 dòng đầu:")
for name, url, room in hotels[:5]:
    print(f"   • {name} — {room}")

📡 Input: Google Sheet online
✅ Đọc được 26 khách sạn. 5 dòng đầu:
   • The Ocean Resort by Fusion Quy Nhon - Studio with Pool View — Studio with Pool View
   • The Ocean Resort by Fusion Quy Nhon - One-bedroom Private Pool Villa - Spa Inclusive — One-bedroom Private Pool Villa - Spa Inclusive
   • The Ocean Resort by Fusion Quy Nhon - Two-bedroom Courtyard Villa - Private Pool — Two-bedroom Courtyard Villa - Private Pool
   • The Ocean Resort by Fusion Quy Nhon - Three-bedroom Garden Villa - Private Pool — Three-bedroom Garden Villa - Private Pool
   • The Ocean Resort by Fusion Quy Nhon - Four-bedroom Beachfront Pool Villa - Ocean Club Access — Four-bedroom Beachfront Pool Villa - Ocean Club Access


In [3]:
# (TÙY CHỌN) Tải Google Sheet về file offline — lần sau chỉ cần đổi INPUT_MODE = "offline"
import pandas as pd
from crawler.hotels_io import _gsheet_url

os.makedirs(os.path.join(ROOT, "input"), exist_ok=True)
dest = OFFLINE_FILE if os.path.isabs(OFFLINE_FILE) else os.path.join(ROOT, OFFLINE_FILE)
pd.read_csv(_gsheet_url(GSHEET_URL)).to_csv(dest, index=False, encoding="utf-8-sig")
print(f"💾 Đã lưu bản offline: {dest}")

💾 Đã lưu bản offline: /Users/hchinhtrung/Documents/GitHub/mvillage-email-template/31.crawl-tool/input/agoda-thy-6.csv


In [4]:
# ════════════════ ③ CRAWL ════════════════
OUTDIR = os.path.join(ROOT, "results", "agoda", RUN_NAME)   # mỗi notebook 1 thư mục kết quả riêng
os.makedirs(OUTDIR, exist_ok=True)
os.chdir(OUTDIR)                     # output (FINAL_*.csv, TEMP_agoda.csv) nằm ở đây

kwargs = dict(
    site="agoda",                    # direct replay (nhanh) + Camoufox warm
    input=INPUT,
    weeks=WEEKS,
    # cache warm dùng CHUNG cho mọi notebook (đỡ warm lại) — chỉ kết quả là tách riêng
    capture_dir=os.path.join(ROOT, "results", "agoda", "captures"),
)
if MAX_HOTELS:
    kwargs["max"] = MAX_HOTELS
if SHARD:
    kwargs["shard"] = SHARD

await crawler.arun(**kwargs)         # notebook cho phép await trực tiếp

📂 Resume: 26 rows from TEMP_agoda.csv
🚀 AGODA crawl | 26 hotels × 6w | direct+fallback | engine=camoufox | W1=2026-09-08
🦊 Camoufox ready (humanize=True geoip=True) — browser navs use anti-detect Firefox
✔️  1/26 The Ocean Resort by Fusion Quy Nhon - Studio with Pool View — complete, skip
✔️  2/26 The Ocean Resort by Fusion Quy Nhon - One-bedroom Private Pool Villa - Spa Inclusive — complete, skip
✔️  3/26 The Ocean Resort by Fusion Quy Nhon - Two-bedroom Courtyard Villa - Private Pool — complete, skip
✔️  4/26 The Ocean Resort by Fusion Quy Nhon - Three-bedroom Garden Villa - Private Pool — complete, skip
✔️  5/26 The Ocean Resort by Fusion Quy Nhon - Four-bedroom Beachfront Pool Villa - Ocean Club Access — complete, skip
✔️  6/26 FLC Quy Nhon Beachfront Condotel — complete, skip
✔️  8/26 FLC Sea Tower Quy Nhơn Seaview Apartment — complete, skip
✔️  9/26 FLC Sea Tower - The Beach Quy Nhon — complete, skip
✔️  10/26 FLC City Hotel Beach Quy Nhon — complete, skip
✔️  11/26 Grand Hyams H

'FINAL_20260903.csv'

In [5]:
# ════════════════ ④ XEM KẾT QUẢ ════════════════
import glob
import pandas as pd

OUTDIR = os.path.join(ROOT, "results", "agoda", RUN_NAME)
files = sorted(glob.glob(os.path.join(OUTDIR, "FINAL_*.csv")))
assert files, "Chưa có file FINAL nào — hãy chạy cell ③ trước."
latest = files[-1]
df = pd.read_csv(latest)
print(f"📄 {latest} — {len(df)} dòng")
df.head(20)

📄 /Users/hchinhtrung/Documents/GitHub/mvillage-email-template/31.crawl-tool/results/agoda/thy-6/FINAL_20260903.csv — 26 dòng


,hotel_name,room_type,price_w1,price_w2,price_w3,price_w4,price_w5,price_w6
0,The Ocean Resort by Fusion Quy Nhon - Studio w...,Studio with Pool View,"3,955,026","3,559,524","3,559,524","3,559,524","3,202,381","3,202,381"
1,The Ocean Resort by Fusion Quy Nhon - One-bedr...,One-bedroom Private Pool Villa - Spa Inclusive,"6,553,923","6,440,414","6,514,335","6,516,121","6,732,804","6,706,507"
2,The Ocean Resort by Fusion Quy Nhon - Two-bedr...,Two-bedroom Courtyard Villa - Private Pool,"8,916,667","8,916,667","8,916,667","8,916,667","8,202,381","8,202,381"
3,The Ocean Resort by Fusion Quy Nhon - Three-be...,Three-bedroom Garden Villa - Private Pool,"14,933,862","13,440,476","13,440,476","13,440,476","13,690,476","13,690,476"
4,The Ocean Resort by Fusion Quy Nhon - Four-bed...,Four-bedroom Beachfront Pool Villa - Ocean Clu...,"41,666,667","37,500,000","37,500,000","37,500,000","32,738,095","32,738,095"
5,FLC Quy Nhon Beachfront Condotel,Junior Deluxe King,"437,978","437,978","437,978","437,978","437,978","437,978"
6,FLC Quy Nhon SeaView Condotel,Sea View Balcony,NaN,NaN,NaN,NaN,NaN,NaN
7,FLC Sea Tower Quy Nhơn Seaview Apartment,Sea View Studio with Balcony,"493,827","439,743","444,760","443,694","448,711","435,793"
8,FLC Sea Tower - The Beach Quy Nhon,Studio Ocean View Twin,"417,636","413,913","414,165","414,112","414,364","413,713"
9,FLC City Hotel Beach Quy Nhon,Deluxe giường đôi Hướng phố (Deluxe Double Cit...,"1,051,001","1,023,063","1,035,015","1,032,474","1,044,426","1,013,652"
